In [1]:
import time
import pandas as pd
import numpy as np
import arff
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
from sklearn.model_selection import train_test_split

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import StratifiedKFold, ParameterGrid
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

warnings.filterwarnings('ignore')
pd.set_option("display.max_colwidth", None)

### Loading data

In [2]:
# Obesity datasets
obesity_df_minmax = pd.read_csv('obesity_df_train_minmax_preprocessed.csv')
obesity_df_zscore = pd.read_csv('obesity_df_train_zscale_preprocessed.csv')

In [3]:
# Depression datasets
depression_df_minmax = pd.read_csv('depression_df_train_minmax_preprocessed.csv')
depression_df_zscore = pd.read_csv('depression_df_train_zscale_preprocessed.csv')

In [6]:
# Amazon reviews datasets
rev_df_minmax = pd.read_csv('rev_df_lrn_minmax_preprocessed.csv')
rev_df_zscore = pd.read_csv('rev_df_lrn_zscale_preprocessed.csv')
print(rev_df_minmax.shape)
print(rev_df_minmax.shape)

(750, 10003)
(750, 10003)


In [5]:
# Congressional voting
voting = pd.read_csv('congressional_voting_preprocessed.csv')

### Functions for training models

In [7]:
# Hyperparams
param_grid = {
    "knn__n_neighbors": range(1, 23, 2),
    "knn__weights": ["uniform", "distance"],
    "knn__metric": ["euclidean", "minkowski", "manhattan"],
    "knn__p": [1, 2],
    "knn__algorithm": ["brute", "auto", "kd_tree", "ball_tree"]
}

In [20]:
def train_knn_with_grid(df, target_col, df_name="dataset", param_grid=param_grid, n_splits=5):
    """
    Train a KNN model using cross-validation across a hyperparameter grid.
    Returns a DataFrame with mean/std metrics for each parameter combination.

    Parameters
    ----------
    df : pandas.DataFrame
        Dataset containing predictors and the target column.
    target_col : str
        Name of the target column to predict.
    df_name : str, optional
        Name of the dataset (used in the output DataFrame).
    param_grid : dict
        Hyperparameter grid to evaluate.
    n_splits : int
        Number of CV folds.
    """

    dataset_name = df_name

    X = df.drop(columns=[target_col]).values
    y_raw = df[target_col].values

    # Encode target labels (required for multiclass ROC-AUC)
    le = LabelEncoder()
    y = le.fit_transform(y_raw)

    grid = list(ParameterGrid(param_grid))
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

    def evaluate_params(params):
        """Evaluate one hyperparameter combination using cross-validation."""
        knn_kwargs = {
            "n_neighbors": params["knn__n_neighbors"],
            "weights": params["knn__weights"],
            "metric": params["knn__metric"],
            "p": params["knn__p"],
            "algorithm": params["knn__algorithm"],
            "n_jobs": -1,
        }

        pipe = Pipeline([
            ("scaler", StandardScaler()),
            ("knn", KNeighborsClassifier(**knn_kwargs)),
        ])

        accs, precs, recs, rocs = [], [], [], []

        start = time.perf_counter()
        for train_idx, valid_idx in cv.split(X, y):
            X_tr, X_va = X[train_idx], X[valid_idx]
            y_tr, y_va = y[train_idx], y[valid_idx]

            try:
                with warnings.catch_warnings():
                    warnings.simplefilter("ignore")
                    pipe.fit(X_tr, y_tr)
                    y_pred = pipe.predict(X_va)
                    y_proba = pipe.predict_proba(X_va)
            except Exception:
                # Skip invalid combinations (e.g., cosine + kd_tree)
                return None

            accs.append(accuracy_score(y_va, y_pred))
            precs.append(precision_score(y_va, y_pred, average="macro", zero_division=0))
            recs.append(recall_score(y_va, y_pred, average="macro", zero_division=0))
            try:
                if y_proba.shape[1] == 2:
                    # Binary case: use positive class probabilities only
                    roc = roc_auc_score(y_va, y_proba[:, 1])
                else:
                    # Multiclass case
                    roc = roc_auc_score(y_va, y_proba, multi_class="ovr", average="macro")
                rocs.append(roc)
            except Exception:
                return None

        elapsed = time.perf_counter() - start

        return {
            "dataset": dataset_name,
            "knn__algorithm": params["knn__algorithm"],
            "knn__metric": params["knn__metric"],
            "knn__n_neighbors": params["knn__n_neighbors"],
            "knn__p": params["knn__p"],
            "knn__weights": params["knn__weights"],
            "precision_mean": np.mean(precs),
            "precision_std": np.std(precs),
            "recall_mean": np.mean(recs),
            "recall_std": np.std(recs),
            "roc_auc_mean": np.mean(rocs),
            "roc_auc_std": np.std(rocs),
            "accuracy_mean": np.mean(accs),
            "accuracy_std": np.std(accs),
            "elapsed_sec": elapsed,
        }

    # Run the grid
    rows = []
    for params in grid:
        res = evaluate_params(params)
        if res is not None:
            rows.append(res)

    results_df = pd.DataFrame(rows).sort_values("roc_auc_mean", ascending=False).reset_index(drop=True)
    return results_df


In [21]:
def select_top5_models(results_df, tolerance=0.005):
    """
    Selects the top 5 hyperparameter combinations for a given model result DataFrame
    based on ROC-AUC (mean), accuracy, stability, and runtime.

    Parameters
    ----------
    results_df : pandas.DataFrame
        The DataFrame returned by train_knn_with_grid().
    tolerance : float, optional (default=0.005)
        Defines how close in roc_auc_mean results should be to be considered "equal".

    Returns
    -------
    pandas.DataFrame
        Top 5 rows sorted by performance and efficiency.
    """

    # keep only models within tolerance from the best roc_auc_mean
    top_mask = results_df["roc_auc_mean"] >= results_df["roc_auc_mean"].max() - tolerance
    best_pool = results_df[top_mask].copy()

    # sort by multiple criterias
    best_df = best_pool.sort_values(
        by=[
            "roc_auc_mean",
            "accuracy_mean",
            "roc_auc_std",
            "elapsed_sec",
            "knn__n_neighbors",
        ],
        ascending=[False, False, True, True, True],
    ).reset_index(drop=True)

    # select top 5
    top5 = best_df.head(5)[[
        "dataset",
        "roc_auc_mean",
        "roc_auc_std",
        "accuracy_mean",
        "precision_mean",
        "elapsed_sec",
        "knn__algorithm",
        "knn__metric",
        "knn__n_neighbors",
        "knn__p",
        "knn__weights",
    ]]

    return top5


### Top-5 models for each df and their settings

In [10]:
results_df_obesity_minmax = train_knn_with_grid(
    df=obesity_df_minmax,
    target_col="obesity_level_grouped",
    df_name="obesity_df_minmax"
)

results_df_obesity_minmax.to_csv('results_df_obesity_minmax.csv', index = False)


top5_results_df_obesity_minmax = select_top5_models(results_df_obesity_minmax)
top5_results_df_obesity_minmax

,dataset,roc_auc_mean,roc_auc_std,accuracy_mean,precision_mean,elapsed_sec,knn__algorithm,knn__metric,knn__n_neighbors,knn__p,knn__weights
0,obesity_df_minmax,0.945234,0.008544,0.793844,0.777239,0.071890,auto,manhattan,17,1,distance
1,obesity_df_minmax,0.945234,0.008544,0.793844,0.777239,0.072597,auto,manhattan,17,2,distance
2,obesity_df_minmax,0.945234,0.008544,0.793844,0.777239,0.074591,brute,minkowski,17,1,distance
3,obesity_df_minmax,0.945234,0.008544,0.793844,0.777239,0.074619,brute,manhattan,17,2,distance
4,obesity_df_minmax,0.945234,0.008544,0.793844,0.777239,0.075749,brute,manhattan,17,1,distance


In [11]:
results_df_obesity_zscore = train_knn_with_grid(
    df=obesity_df_minmax,
    target_col="obesity_level_grouped",
    df_name="obesity_df_zscore"
)

results_df_obesity_zscore.to_csv('results_df_obesity_zscore.csv', index = False)

top5_results_df_obesity_zscore = select_top5_models(results_df_obesity_zscore)
top5_results_df_obesity_zscore

,dataset,roc_auc_mean,roc_auc_std,accuracy_mean,precision_mean,elapsed_sec,knn__algorithm,knn__metric,knn__n_neighbors,knn__p,knn__weights
0,obesity_df_zscore,0.945234,0.008544,0.793844,0.777239,0.071848,auto,manhattan,17,2,distance
1,obesity_df_zscore,0.945234,0.008544,0.793844,0.777239,0.072712,auto,minkowski,17,1,distance
2,obesity_df_zscore,0.945234,0.008544,0.793844,0.777239,0.073326,brute,minkowski,17,1,distance
3,obesity_df_zscore,0.945234,0.008544,0.793844,0.777239,0.074265,brute,manhattan,17,1,distance
4,obesity_df_zscore,0.945234,0.008544,0.793844,0.777239,0.075121,auto,manhattan,17,1,distance


In [ ]:
results_df_depression_minmax = train_knn_with_grid(
    df=depression_df_minmax,
    target_col="depression",
    df_name="depression_df_minmax"
)

results_df_depression_minmax.to_csv('results_df_depression_minmax.csv', index = False)

top5_results_df_depression_minmax = select_top5_models(results_df_depression_minmax)
top5_results_df_depression_minmax

In [31]:
results_df_depression_zscore = train_knn_with_grid(
    df=depression_df_zscore,
    target_col="depression",
    df_name="depression_df_zscore"
)

results_df_depression_zscore.to_csv('results_df_depression_zscore.csv', index = False)

top5_results_df_depression_zscore = select_top5_models(results_df_depression_zscore)
top5_results_df_depression_zscore

In [24]:
results_df_voting = train_knn_with_grid(
    df=voting,
    target_col="class",
    df_name="voting"
)

results_df_voting.to_csv('results_df_voting.csv', index = False)

top5_results_df_voting = select_top5_models(results_df_voting)
top5_results_df_voting

,dataset,roc_auc_mean,roc_auc_std,accuracy_mean,precision_mean,elapsed_sec,knn__algorithm,knn__metric,knn__n_neighbors,knn__p,knn__weights
0,voting,0.987043,0.007909,0.926744,0.92585,0.032443,brute,minkowski,7,2,uniform
1,voting,0.987043,0.007909,0.926744,0.92585,0.035268,auto,minkowski,7,2,uniform
2,voting,0.987043,0.007909,0.926744,0.92585,0.040609,brute,euclidean,7,1,uniform
3,voting,0.987043,0.007909,0.926744,0.92585,0.040613,brute,euclidean,7,2,uniform
4,voting,0.987043,0.007909,0.926744,0.92585,0.042774,auto,euclidean,7,2,uniform


In [16]:
results_df_rev_minmax = train_knn_with_grid(
    df=rev_df_minmax,
    target_col="Class_encoded",
    df_name="rev_df_minmax"
)

results_df_rev_minmax.to_csv('results_df_rev_minmax.csv', index = False)

top5_results_df_rev_minmax = select_top5_models(results_df_rev_minmax)
top5_results_df_rev_minmax

In [14]:
results_df_rev_zscale = train_knn_with_grid(
    df=rev_df_zscore,
    target_col="Class_encoded",
    df_name="rev_df_zscore"
)


results_df_rev_zscale.to_csv('results_df_rev_zscale.csv', index = False)

top5_results_df_rev_zscale = select_top5_models(results_df_rev_zscale)
top5_results_df_rev_zscale

,dataset,roc_auc_mean,roc_auc_std,accuracy_mean,precision_mean,elapsed_sec,knn__algorithm,knn__metric,knn__n_neighbors,knn__p,knn__weights
0,rev_df_zscore,0.642093,0.020149,0.062667,0.101871,4.138027,ball_tree,manhattan,21,2,distance
1,rev_df_zscore,0.642093,0.020149,0.062667,0.101871,4.155081,ball_tree,minkowski,21,1,distance
2,rev_df_zscore,0.642093,0.020149,0.062667,0.101871,4.227876,ball_tree,manhattan,21,1,distance
3,rev_df_zscore,0.642093,0.020149,0.062667,0.101871,5.136090,kd_tree,manhattan,21,1,distance
4,rev_df_zscore,0.642093,0.020149,0.062667,0.101871,5.164446,kd_tree,minkowski,21,1,distance
